In [1]:
import sys
sys.path.append("../../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from simulator.simulation.modules import Campaign, History
from simulator.simulation.simulate import simulate_campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.model.rlb_dp_bidder import RLBDPBidder
from simulator.validation.check_results import autobidder_check

In [2]:
# Load data
auction_mode = 'FPA'
campaigns_path = '../../data/small_example/fpa/subsample_campaigns.csv'
stats_path = '../../data/small_example/fpa/subsample_stats.csv'

campaign_df = pd.read_csv(campaigns_path)
stats_df = pd.read_csv(stats_path)

print(f"Loaded {len(campaign_df)} campaigns")
print(f"Loaded {len(stats_df):,} stat records")

Loaded 10000 campaigns
Loaded 9,210,861 stat records


In [3]:
CUSTOM_PARAMS = {
    'max_bid': 300,
    'gamma': 1.0,  # Discount factor (typically 1.0 for undiscounted)
    'model_path': None,
    'N_bound': 24,  # Max hours for normalization
    'B_bound': 10000,  # Max budget for normalization
}
MODEL_PATH = 'rlb_dp_model_new.pkl'

rlb_dp_bidder = RLBDPBidder(CUSTOM_PARAMS)
rlb_dp_bidder.fit(stats_df)
rlb_dp_bidder.save_model(MODEL_PATH)

Hours: 100%|██████████| 24/24 [00:00<00:00, 39.69it/s]


In [4]:
# Pick a campaign to test
CAMPAIGN_IDX = 25  # You can change this

campaign_data = campaign_df.iloc[CAMPAIGN_IDX]

print(f"Campaign {campaign_data['campaign_id']}:")
print(f"  Budget: ${campaign_data['auction_budget']:.2f}")
print(f"  Duration: {(campaign_data['campaign_end'] - campaign_data['campaign_start'])/3600:.1f} hours")
print(f"  Category: {campaign_data['logical_category']}")
print(f"  Region: {campaign_data['region_id']}")

Campaign 86755692:
  Budget: $195.84
  Duration: 24.0 hours
  Category: 5.26
  Region: 643030


In [5]:
# Create campaign object
campaign_example = Campaign(
    item_id=campaign_data['item_id'],
    campaign_id=int(campaign_data['campaign_id']),
    loc_id=int(campaign_data["loc_id"]),
    region_id=int(campaign_data["region_id"]),
    logical_category=campaign_data["logical_category"],
    microcat_ext=int(campaign_data["microcat_ext"]),
    campaign_start=int(campaign_data["campaign_start"]),
    campaign_end=int(campaign_data["campaign_end"]),
    initial_balance=campaign_data['auction_budget'],
    balance=campaign_data['auction_budget'],
    curr_time=int(campaign_data["campaign_start"]),
    prev_time=int(campaign_data["campaign_start"]),
    prev_balance=campaign_data['auction_budget'],
)

### Демонстрация бага: запуск simulate дважды с одной campaign

Запусти эту ячейку и следующую дважды (без перезапуска campaign_example). При втором запуске увидишь `!!! ОШИБКА` — campaign уже израсходован (balance=0).

In [6]:
# Демо бага: один и тот же campaign_example, два запуска подряд
print("=== ПЕРВЫЙ запуск (campaign свежая) ===")
h1 = simulate_campaign(campaign=campaign_example, bidder=rlb_dp_bidder, stats_file=stats_df, auction_mode=auction_mode)
print(f"  Steps: {len(h1.rows)}, balance после: {campaign_example.balance:.2f}")
print()
print("=== ВТОРОЙ запуск (campaign уже мутирована!) ===")
h2 = simulate_campaign(campaign=campaign_example, bidder=rlb_dp_bidder, stats_file=stats_df, auction_mode=auction_mode)
print(f"  Steps: {len(h2.rows)}, balance после: {campaign_example.balance:.2f}")

=== ПЕРВЫЙ запуск (campaign свежая) ===
[simulate_campaign] ВХОД: campaign_id=86755692 balance=195.84 initial_balance=195.84 clicks=0.00 curr_time=530521269
  Steps: 25, balance после: 70.25

=== ВТОРОЙ запуск (campaign уже мутирована!) ===
[simulate_campaign] ВХОД: campaign_id=86755692 balance=70.25 initial_balance=195.84 clicks=43.07 curr_time=530611200
  Steps: 25, balance после: 44.49


In [7]:
# Run simulation
print("Running simulation...")
simulation_history = simulate_campaign(
    campaign=campaign_example,
    bidder=rlb_dp_bidder,
    stats_file=stats_df,
    auction_mode=auction_mode
)
print("✓ Simulation complete!")

Running simulation...
[simulate_campaign] ВХОД: campaign_id=86755692 balance=44.49 initial_balance=195.84 clicks=85.09 curr_time=530611200
✓ Simulation complete!


In [8]:
# Get results
df = simulation_history.to_data_frame()

final_clicks = df['clicks'].iloc[-1]
final_balance = df['balance'].iloc[-1]
total_spend = campaign_data['auction_budget'] - final_balance
budget_utilization = total_spend / campaign_data['auction_budget'] * 100

print("="*60)
print("CAMPAIGN RESULTS")
print("="*60)
print(f"Initial budget:       ${campaign_data['auction_budget']:.2f}")
print(f"Total spend:          ${total_spend:.2f}")
print(f"Final balance:        ${final_balance:.2f}")
print(f"Budget utilization:   {budget_utilization:.1f}%")
print()
print(f"Total clicks:         {final_clicks:.2f}")

if total_spend > 0 and final_clicks > 0:
    cost_per_click = total_spend / final_clicks
    print(f"Cost per click:       ${cost_per_click:.2f}")

print()
print(f"Simulation steps:     {len(df)}")
print(f"Average bid:          ${df['bid'].mean():.2f}")
print(f"Min bid:              ${df['bid'].min():.2f}")
print(f"Max bid:              ${df['bid'].max():.2f}")

CAMPAIGN RESULTS
Initial budget:       $195.84
Total spend:          $177.11
Final balance:        $18.73
Budget utilization:   90.4%

Total clicks:         127.11
Cost per click:       $1.39

Simulation steps:     25
Average bid:          $10.00
Min bid:              $10.00
Max bid:              $10.00


### Старая модель

In [9]:
rlb_dp_bidder_old  = RLBDPBidder(params={
    'model_path': '../../data/rlb_dp_model.pkl',
    'max_bid': 300,
})

print("✓ OLD RLB-DP bidder created")

✓ OLD RLB-DP bidder created


In [10]:
# Run simulation
print("Running simulation...")
simulation_history = simulate_campaign(
    campaign=campaign_example,
    bidder=rlb_dp_bidder_old,
    stats_file=stats_df,
    auction_mode=auction_mode
)
print("✓ Simulation complete!")

Running simulation...
[simulate_campaign] ВХОД: campaign_id=86755692 balance=18.73 initial_balance=195.84 clicks=127.11 curr_time=530611200
✓ Simulation complete!


In [11]:
# Get results
df = simulation_history.to_data_frame()

final_clicks = df['clicks'].iloc[-1]
final_balance = df['balance'].iloc[-1]
total_spend = campaign_data['auction_budget'] - final_balance
budget_utilization = total_spend / campaign_data['auction_budget'] * 100

print("="*60)
print("CAMPAIGN RESULTS")
print("="*60)
print(f"Initial budget:       ${campaign_data['auction_budget']:.2f}")
print(f"Total spend:          ${total_spend:.2f}")
print(f"Final balance:        ${final_balance:.2f}")
print(f"Budget utilization:   {budget_utilization:.1f}%")
print()
print(f"Total clicks:         {final_clicks:.2f}")

if total_spend > 0 and final_clicks > 0:
    cost_per_click = total_spend / final_clicks
    print(f"Cost per click:       ${cost_per_click:.2f}")

print()
print(f"Simulation steps:     {len(df)}")
print(f"Average bid:          ${df['bid'].mean():.2f}")
print(f"Min bid:              ${df['bid'].min():.2f}")
print(f"Max bid:              ${df['bid'].max():.2f}")

CAMPAIGN RESULTS
Initial budget:       $195.84
Total spend:          $193.07
Final balance:        $2.77
Budget utilization:   98.6%

Total clicks:         159.62
Cost per click:       $1.21

Simulation steps:     25
Average bid:          $5.03
Min bid:              $2.77
Max bid:              $10.00


# На всех компаниях

In [13]:
campaigns_path = '../../data/small_example/fpa/subsample_campaigns.csv'
stats_path = '../../data/small_example/fpa/subsample_stats.csv'
    
results = autobidder_check(
    bidder=RLBDPBidder,
    params={
        "input_campaigns": campaigns_path,
        "input_stats": stats_path,
        'model_path': MODEL_PATH,
        'max_bid': 300,
    },
    auction_mode='FPA'
)

[simulate_campaign] ВХОД: campaign_id=76801828 balance=307.20 initial_balance=307.20 clicks=0.00 curr_time=529789799


In [14]:
results

{'status': 'OK!',
 'status_msg': '',
 'time_overall_sec': 7.435101747512817,
 'time_inference_sec': 0.16276097297668457,
 'score': (np.float64(1.7840409092981842),
  np.float64(1.3153659275881564),
  np.float64(112.12442013587851),
  0.0)}

In [15]:
campaigns_path = '../../data/small_example/fpa/subsample_campaigns.csv'
stats_path = '../../data/small_example/fpa/subsample_stats.csv'
    
results = autobidder_check(
    bidder=RLBDPBidder,
    params={
        "input_campaigns": campaigns_path,
        "input_stats": stats_path,
        'model_path': '../../data/rlb_dp_model.pkl',
        'max_bid': 300,
    },
    auction_mode='FPA'
)

[simulate_campaign] ВХОД: campaign_id=76801828 balance=307.20 initial_balance=307.20 clicks=0.00 curr_time=529789799


In [16]:
results

{'status': 'OK!',
 'status_msg': '',
 'time_overall_sec': 6.785645961761475,
 'time_inference_sec': 0.16193938255310059,
 'score': (np.float64(0.6293452833953792),
  np.float64(1.3933199941414285),
  np.float64(109.02023420171025),
  0.0)}